# Description

This notebook is used for a one time task of finding emails and profiles of list from AI VC list

In [2]:
#!sudo /bin/bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

     |████████████████████████████████| 12.0 MB 9.2 MB/s eta 0:00:01                | 5.8 MB 9.2 MB/s eta 0:00:01�████▌| 11.8 MB 9.2 MB/s eta 0:00:01
     |████████████████████████████████| 139 kB 64.4 MB/s eta 0:00:01
     |████████████████████████████████| 96 kB 5.6 MB/s  eta 0:00:01
     |████████████████████████████████| 50 kB 6.7 MB/s  eta 0:00:01
     |████████████████████████████████| 220 kB 28.0 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 77.6 MB/s eta 0:00:01


In [73]:
import logging
import os

import numpy as np
import pandas as pd

import ck_marketing.dropcontact.dropcontact_api as cmdrdrap
import ck_marketing.hunterio.hunter_api as cmhuhuap
import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint
from ck_marketing.hunterio.hunter_api import GoogleSheetsHelper, HunterIO
from ck_marketing.linkedin.phantombuster_api import Phantom

In [7]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Print system signature.
_LOG.info("%s", henv.get_system_signature()[0])

# Configure the notebook style.
hprint.config_notebook()

DEBUG:helpers.hsystem:> (cd . && cd "$(git rev-parse --show-toplevel)/.." && (git rev-parse --is-inside-work-tree | grep -q true)) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --show-toplevel) 2>&1
DEBUG:helpers.hsystem:> (git branch --show-current) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --short HEAD) 2>&1
DEBUG:helpers.hsystem:> (git log --date=local --oneline --graph --date-order --decorate --pretty=format:'%h %<(8)%aN%  %<(65)%s (%>(14)%ar) %ad %<(10)%d' -3) 2>&1


INFO:__main__:# Git
  branch_name='CmTask9112_AI_VC_list_from_David'
  hash='3dfabaa81'
  # Last commits:
    * 3dfabaa81 Vedanshu Joshi CmTask9087 Create a changelog file for binance (#9092)            (40 minutes ago) Fri Jul 19 13:02:19 2024  (HEAD -> CmTask9112_AI_VC_list_from_David, origin/master, origin/HEAD, master)
    * 3b7f1a017 Vlad     CmampTask8917_Compute_annualized_alpha (#9037)                    (  18 hours ago) Thu Jul 18 20:09:01 2024           
    * 4cb12055a Vlad     CmampTask8918_upload_xls_file (#9074)                             (  19 hours ago) Thu Jul 18 19:03:40 2024           
# Machine info
  system=Linux
  node name=8951c6869c15
  release=5.15.0-1056-aws
  version=#61~20.04.1-Ubuntu SMP Wed Mar 13 17:40:41 UTC 2024
  machine=x86_64
  processor=x86_64
  cpu count=8
  cpu freq=scpufreq(current=2499.998, min=0.0, max=0.0)
  memory=svmem(total=33280270336, available=21531013120, percent=35.3, used=10901303296, free=5508100096, active=7012327424, inactive=8784

In [47]:
hunter_api_key = os.getenv("Hunter_API_KEY")
dropcontact_api_key = os.getenv("DropContact_API_KEY")
phantom_api_key = os.getenv("Phantom_API_KEY")

In [10]:
google_creds_path = "service.json"
google_sheet_helper = GoogleSheetsHelper(google_creds_path)

In [14]:
file_id = "1Xh-DwrEQ0sLfV03GMgndeKIXrdcDTwSLl0SXFBEoX5w"

In [15]:
df = google_sheet_helper.read_sheet(file_id)

In [16]:
df_split = (
    df.set_index(["Venture Fund", "Companies invested"])["Investor"]
    .str.split(", ", expand=True)
    .stack()
    .reset_index(level=2, drop=True)
    .reset_index(name="Investor")
)

In [19]:
df_split_2 = df_split

In [20]:
df_split_2["firstName"] = df_split["Investor"].str.split().str[0]
df_split_2["lastName"] = df_split["Investor"].str.split().str[-1]

In [21]:
sheet = google_sheet_helper.google_account.open_by_key(file_id)
cleaned_profiles_tab = sheet.add_worksheet(
    title="cleaned_profiles_1", rows="100", cols="20"
)
cleaned_profiles_tab_2 = sheet.add_worksheet(
    title="cleaned_profiles_2", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, df_split, "cleaned_profiles_1")
google_sheet_helper.write_results(file_id, df_split_2, "cleaned_profiles_2")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: cleaned_profiles_2


# HunterIO - Extract emails

In [ ]:
first_name_col = "firstName"
last_name_col = "lastName"
company_col = "Venture Fund"
tab_name = "cleaned_profiles_2"

cmhuhuap.process_records(
    api_key=hunter_api_key,
    google_creds_path=google_creds_path,
    file_id=file_id,
    first_name_col=first_name_col,
    last_name_col=last_name_col,
    company_col=company_col,
    tab_name=tab_name,
)

# DropContact - Extract Remaining emails


In [24]:
df_drop = google_sheet_helper.read_sheet(file_id, "hunter_results")

In [25]:
missing_email_df = df_drop[
    df_drop["hunter_extracted_email"].isna()
    | (df_drop["hunter_extracted_email"] == "")
]

In [29]:
# Prepare data for DropContact
first_names = missing_email_df["firstName"].tolist()
last_names = missing_email_df["lastName"].tolist()
company_names = missing_email_df["Venture Fund"].tolist()

# Get emails from DropContact
dropcontact_results_df = cmdrdrap.get_email_from_dropcontact(
    first_names, last_names, company_names, dropcontact_api_key
)

Processing batches:   0%|                                                     | 0/5 [00:00<?, ?it/s]

Starting query batch 0.
Batch 0: Query ID: bedtoyrczquwmor.


Processing batches:  20%|#########                                    | 1/5 [01:26<05:44, 86.09s/it]

Batch 0: Query finished. Credits left: 1020.
Batch 0 completed in 86.09 seconds.
Starting query batch 1.
Batch 1: Query ID: hhyqhygbwhlesup.


Processing batches:  40%|##################                           | 2/5 [03:01<04:35, 91.77s/it]

...

Batch 2: Query failed, reason: timeout.
Batch 2 completed in 127.07 seconds.
Starting query batch 3.
Batch 3: Query ID: rrccdtgeiksdqpc.


Processing batches:  80%|###################################2        | 4/5 [07:05<01:51, 111.37s/it]

Batch 3: Query finished. Credits left: 870.
Batch 3 completed in 116.70 seconds.
Starting query batch 4.
Batch 4: Query ID: rawppwzrnqvhhje.


Processing batches: 100%|############################################| 5/5 [09:12<00:00, 110.41s/it]

Batch 4: Query failed, reason: timeout.
Batch 4 completed in 126.43 seconds.
Total processing time: 552.05 seconds.


In [30]:
# Replace special float values with NaN
df_drop.replace([np.inf, -np.inf], np.nan, inplace=True)
dropcontact_results_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Rename columns in dropcontact_results_df to match df_drop
dropcontact_results_df.rename(
    columns={"first name": "firstName", "last name": "lastName"}, inplace=True
)

# Merge the dataframes on 'firstNames' and 'lastName', keeping only the 'email' column from dropcontact_results_df
merged_df = pd.merge(
    df_drop,
    dropcontact_results_df[["firstName", "lastName", "email"]],
    on=["firstName", "lastName"],
    how="left",
)

# Rename the 'email' column to 'dropcontact_mail'
merged_df.rename(columns={"email": "dropcontact_mail"}, inplace=True)

# Count the non-null values in the 'dropcontact_mail' column
email_count = merged_df[
    merged_df["dropcontact_mail"].notna() & (merged_df["dropcontact_mail"] != "")
]["dropcontact_mail"].count()
print(f"Number of emails found: {email_count}")
print(f"Number of profile emails hunter could not find: {len(missing_email_df)}")

Number of emails found: 62
Number of profile emails hunter could not find: 243


In [31]:
merged_df.replace({np.nan: "", np.inf: "", -np.inf: ""}, inplace=True)
cleaned_profiles_tab = sheet.add_worksheet(
    title="hunter_drop_emails", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, merged_df, "hunter_drop_emails")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: hunter_drop_emails


In [32]:
merged_df["all_emails"] = (
    merged_df["hunter_extracted_email"]
    .fillna("")
    .replace("", pd.NA)
    .combine_first(merged_df["dropcontact_mail"])
)
merged_df.replace({np.nan: "", np.inf: "", -np.inf: ""}, inplace=True)

In [33]:
cleaned_profiles_tab = sheet.add_worksheet(
    title="all_emails", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, merged_df, "all_emails")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: all_emails


# HunterIO - Verify emails


In [34]:
hunter_instance = HunterIO(hunter_api_key)
verified_df = hunter_instance.verify_emails(merged_df, "all_emails")

In [35]:
cleaned_profiles_tab = sheet.add_worksheet(
    title="hunter_verification", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, verified_df, "hunter_verification")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: hunter_verification


In [39]:
final_df = verified_df[
    [
        "firstName",
        "lastName",
        "all_emails",
        "Venture Fund",
        "hunter_verification",
    ]
]
# Step 2: Filter out rows where 'hunter_extracted_email' is empty.
final_df = final_df[
    final_df["all_emails"].notna() & (final_df["all_emails"] != "")
]

cleaned_profiles_tab = sheet.add_worksheet(
    title="found_and_verified_mails", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, final_df, "found_and_verified_mails")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: found_and_verified_mails


# PhantomBustor - Extract linkedin

In [74]:
# Initialize the Phantom instance.
phantom = Phantom(phantom_api_key)
# Get and print all agents and their IDs.
agents = phantom.get_all_agents()
print("List of all agents and their IDs:\n")
for agent in agents:
    print(f"Agent Name: {agent['name']}, Agent ID: {agent['id']}")

List of all agents and their IDs:

Agent Name: Venture Fund David, Agent ID: 2580268935041081
Agent Name: Venture Funds, Agent ID: 3093093291849192
Agent Name: Untitled LinkedIn Profile Scraper, Agent ID: 4221221321911511
Agent Name: Sequoia Capital Trual, Agent ID: 2595459631539241


In [76]:
# Get agent ID and name.
AGENT_ID = "2580268935041081"
specific_agent_name = phantom.get_agent_name(AGENT_ID)
print(f"Selected Phantom: {specific_agent_name}")
# Launch the agent and get the results in a DataFrame.
df = phantom.launch_and_get_df(AGENT_ID)
print("DataFrame is fetched")

Selected Phantom: Venture Fund David
{'containerId': '8641879303297297'}
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent still running, waiting for 30 seconds...
wait..
Agent sti

In [77]:
df.head(2)

,firstName,lastName,Venture Fund,all_emails,hunter_verification,query,timestamp,url,description,title,error
0,Danielle,Strachman,1517 Fund,danielle@1517fund.com,accept_all,Danielle Strachman 1517 Fund,2024-07-19T16:05:29.126Z,https://linkedin.com/in/daniellestrachman,The Thiel Foundation Graphic. Senior Advisor t...,Danielle Strachman - San Francisco Bay Area,
1,Michael,Gibson,1517 Fund,michael@1517fund.com,accept_all,Michael Gibson 1517 Fund,2024-07-19T16:05:40.772Z,https://linkedin.com/in/michael-patrick-gibson...,Location: Telluride · 500+ connections on Link...,"Michael Patrick Gibson - Telluride, Colorado, ...",


In [78]:
cleaned_profiles_tab = sheet.add_worksheet(
    title="linkedin_profile_extraction", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, df, "linkedin_profile_extraction")

/venv/lib/python3.9/site-packages/gspread/worksheet.py:1069: UserWarning: [Deprecated][in version 6.0.0]: method signature will change to: 'Worksheet.update(value = [[]], range_name=)' arguments 'range_name' and 'values' will swap, values will be mandatory of type: 'list(list(...))'
  warnings.warn(
INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: linkedin_profile_extraction
